# Function 4 Analysis - Week 7

**Function Description:** Address the challenge of optimally placing products across warehouses for a business with high online sales, where accurate calculations are costly and only feasible biweekly. To speed up decision-making, an ML model approximates these results within hours. The model has four hyperparameters to tune, and its output reflects the difference from the expensive baseline. Because the system is dynamic and full of local optima, it requires careful tuning and robust validation to find reliable, near-optimal solutions.

**New datapoint (Week 7):** `0.421100-0.389600-0.370500-0.393100` returned **≈0.37109** — strong but below the current max `0.430300-0.359300-0.351800-0.383700` at **≈0.55184**. Total observations: **36**.

**Why we chose this point:** We stayed in the high-mean pocket but moved slightly off the peak to test stability; the lower score confirms the spike is narrow around the incumbent.

**Recommendation for next week:** Keep EI local around the max; use a tight box roughly x1≈0.41–0.44, x2≈0.34–0.40, x3≈0.33–0.38, x4≈0.37–0.41 with light jitter/diversity to avoid re-sampling the exact max. Allow small steps only; prioritise candidates that stay near the incumbent while probing its immediate neighborhood.


## Loading and Displaying the Data

We load the inputs and outputs for function 4. The Week 7 run `(0.430300, 0.359300, 0.351800, 0.383700)` returned **≈0.5518 (new max)**, beating the prior best Week 3 point `(0.4262, 0.4527, 0.3919, 0.4293)` at ≈−0.0140. Earlier points: Week 1 manual override missed (≈−11.6), Week 2 UCB found ≈−0.058, Week 4 follow-up was ≈−0.100.


In [22]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from IPython.display import display
sns.set_theme(style="ticks", context="notebook")
path = Path("../../initial_data/function_4")
X = np.load(path / "initial_inputs.npy")
y = np.load(path / "initial_outputs.npy")

# Week 1–6 new points
X_new_point_week_1 = np.array([[0.100000, 0.400000, 0.400000, 0.050000]])
y_new_point_week_1 = np.array([-11.551402216263181])
X_new_point_week_2 = np.array([[0.412000, 0.448200, 0.386300, 0.439500]])
y_new_point_week_2 = np.array([-0.05797573871593498])
X_new_point_week_3 = np.array([[0.426200, 0.452700, 0.391900, 0.429300]])
y_new_point_week_3 = np.array([-0.013999616551390925])
X_new_point_week_4 = np.array([[0.430000, 0.455200, 0.393800, 0.427600]])
y_new_point_week_4 = np.array([-0.09998342305973962])
X_new_point_week_5 = np.array([[0.430300, 0.359300, 0.351800, 0.383700]])
y_new_point_week_5 = np.array([0.5518426262369016])
X_new_point_week_6 = np.array([[0.421100, 0.389600, 0.370500, 0.393100]])
y_new_point_week_6 = np.array([0.37109387744135747])

X = np.vstack([
    X,
    X_new_point_week_1,
    X_new_point_week_2,
    X_new_point_week_3,
    X_new_point_week_4,
    X_new_point_week_5,
    X_new_point_week_6,
])
y = np.concatenate([
    y,
    y_new_point_week_1,
    y_new_point_week_2,
    y_new_point_week_3,
    y_new_point_week_4,
    y_new_point_week_5,
    y_new_point_week_6,
])

df = pd.DataFrame(X, columns=["x1", "x2", "x3", "x4"])
df["y"] = y

# Display original and y-sorted DataFrames side by side
display(df)
print("df sorted by y")
df_sorted = df.sort_values("y", ascending=False).reset_index(drop=True)
df_sorted["x_avg"] = df_sorted[["x1", "x2", "x3", "x4"]].mean(axis=1)
display(df_sorted)


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


df sorted by y


,x1,x2,x3,x4,y,x_avg
0,0.430300,0.359300,0.351800,0.383700,0.551843,0.381275
1,0.421100,0.389600,0.370500,0.393100,0.371094,0.393575
2,0.426200,0.452700,0.391900,0.429300,-0.014000,0.425025
3,0.412000,0.448200,0.386300,0.439500,-0.057976,0.421500
4,0.430000,0.455200,0.393800,0.427600,-0.099983,0.426650
5,0.577766,0.428772,0.425826,0.249007,-4.025542,0.420343
6,0.326076,0.472367,0.453192,0.105887,-6.702089,0.339381
7,0.282138,0.505987,0.530531,0.096302,-7.966775,0.353739
8,0.124871,0.129770,0.384400,0.287076,-10.069633,0.231529
9,0.100000,0.400000,0.400000,0.050000,-11.551402,0.237500


- **New point (Week 7):** `(0.421100, 0.389600, 0.370500, 0.393100)` returned **≈0.3711** — strong but **still below** last week’s 0.5518 peak at `(0.430300, 0.359300, 0.351800, 0.383700)`.
- Recommendation for next BO step: keep the local EI focus around `(0.43, 0.36–0.39, 0.35–0.40)` with a **narrow box constraint** and a noise term; add a diversity/jitter penalty so proposals don’t snap back exactly to the 0.5518 point but explore its immediate neighborhood (±0.02 per coord).



In [23]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
np.random.seed(42)
kernel = Matern(length_scale=0.8, nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, n_restarts_optimizer=5)
gp.fit(X, y)
print("GP fitted successfully")


GP fitted successfully


## Finding the Next Point to Evaluate (Pure Exploitation)

Given the strong result from Week 2 (≈−0.058, very close to zero), we now switch to **pure exploitation** by maximizing the GP mean prediction `μ(x)` directly, rather than using UCB. This focuses our search on refining the estimate of the optimal region around the newly found maximum, with no exploration component. We still include a small penalty term to favour moderate x values on average, but the primary driver is the GP mean prediction.

In [24]:
# Local EI with drift penalty (stay near incumbent, discourage x2/x3 drift)
from scipy.special import erf

best_idx = df["y"].idxmax()
best_point = df.loc[best_idx, ["x1", "x2", "x3", "x4", "y"]]

# Tight local box around the incumbent (matches the recommendation ranges)
x1_grid = np.linspace(0.41, 0.44, 40)
x2_grid = np.linspace(0.34, 0.40, 40)
x3_grid = np.linspace(0.33, 0.38, 40)
x4_grid = np.linspace(0.37, 0.41, 30)
mesh = np.array(np.meshgrid(x1_grid, x2_grid, x3_grid, x4_grid)).reshape(4, -1).T

mu, sigma = gp.predict(mesh, return_std=True)
y_best = y.max()
xi = 0.003  # mild exploration

sigma_safe = np.maximum(sigma, 1e-9)
sqrt2 = np.sqrt(2.0)
sqrt2pi = np.sqrt(2 * np.pi)
z = (mu - y_best - xi) / sigma_safe
ei = (mu - y_best - xi) * 0.5 * (1.0 + erf(z / sqrt2)) + sigma_safe * (np.exp(-0.5 * z**2) / sqrt2pi)
ei[sigma <= 1e-9] = 0.0

# Drift penalty on x2/x3 away from incumbent
inc_x2x3 = best_point[["x2", "x3"]].to_numpy()
inc_x2x3 = inc_x2x3.astype(float)
drift = np.linalg.norm(mesh[:, 1:3] - inc_x2x3, axis=1)
lambda_pen = 0.1  # small weight; larger means stronger penalty
penalised_ei = ei - lambda_pen * drift

cand = pd.DataFrame(mesh, columns=["x1", "x2", "x3", "x4"])
cand["mu"], cand["sigma"], cand["ei"], cand["drift"], cand["ei_pen"] = mu, sigma, ei, drift, penalised_ei

next_point = cand.loc[cand["ei_pen"].idxmax()]
print("Current best (observed):", best_point.to_dict())
print("Suggested next query (EI with drift penalty): {}-{}-{}-{}".format(
    f"{next_point.x1:.6f}", f"{next_point.x2:.6f}", f"{next_point.x3:.6f}", f"{next_point.x4:.6f}"
))
print(f"Posterior mean: {next_point.mu:.4f}, std: {next_point.sigma:.4f}, EI: {next_point.ei:.6f}, EI_pen: {next_point.ei_pen:.6f}, drift: {next_point.drift:.4f}")

# Show top 5 penalised EI candidates
print("\nTop 5 (penalised EI):")
display(cand.nlargest(5, "ei_pen")[["x1", "x2", "x3", "x4", "mu", "sigma", "ei", "ei_pen", "drift"]])


Current best (observed): {'x1': 0.4303, 'x2': 0.3593, 'x3': 0.3518, 'x4': 0.3837, 'y': 0.5518426262369016}
Suggested next query (EI with drift penalty): 0.440000-0.390769-0.330000-0.410000
Posterior mean: 0.9984, std: 0.2929, EI: 0.451895, EI_pen: 0.448066, drift: 0.0383

Top 5 (penalised EI):


,x1,x2,x3,x4,mu,sigma,ei,ei_pen,drift
1630829,0.44,0.390769,0.33,0.41,0.998430,0.292917,0.451895,0.448066,0.038283
1582829,0.44,0.389231,0.33,0.41,0.998966,0.287536,0.451729,0.448027,0.037028
1678829,0.44,0.392308,0.33,0.41,0.997518,0.298320,0.451744,0.447788,0.039557
1534829,0.44,0.387692,0.33,0.41,0.999124,0.282180,0.451245,0.447665,0.035796
1726829,0.44,0.393846,0.33,0.41,0.996228,0.303739,0.451281,0.447197,0.040849


### Recommendation and context
- Current best (y ≈ 0.551843): `0.430300-0.359300-0.351800-0.383700`
- Proposed next point: `0.440000-0.390769-0.330000-0.410000` (local EI with drift penalty on x2/x3)
- Next 5 alternatives (context, best-first after incumbent):
  - `0.421100-0.389600-0.370500-0.393100` → ≈0.37109
  - `0.426200-0.452700-0.391900-0.429300` → ≈−0.01400
  - `0.412000-0.448200-0.386300-0.439500` → ≈−0.05798
  - `0.430000-0.455200-0.393800-0.427600` → ≈−0.09998
  - `0.577766-0.428772-0.425826-0.249007` → ≈−4.02554

### What changed and why
Switched from pure mean exploitation to EI with a **drift penalty on x2/x3** and a tight local box. EI keeps mild exploration (ξ≈0.003) while the penalty downweights moves that stray laterally from the incumbent, encouraging checks in the immediate neighborhood of the max instead of drifting toward the broader cluster.

